In [ ]:
%env SPDLOG_LEVEL=warning

In [ ]:
from benchmark_utils import benchmark_with_percentiles
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import numpy as np
import python_kernel

In [ ]:
N = 1024 * 1024 * 100
inHost = np.arange(N, dtype=np.float32)

inDevice = python_kernel.create_device_buffer(inHost)
rDevice = python_kernel.create_device_buffer(inHost.size * inHost.itemsize)

In [ ]:
workgroupSizesX = [32, 64, 128, 256, 512, 1024]

p0_values = []
p5_values = []

for size in workgroupSizesX:
    percentiles = benchmark_with_percentiles(lambda: python_kernel.minmax(inDevice, rDevice, size))

    p0_values.append(percentiles[0])
    p5_values.append(percentiles[5])

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(workgroupSizesX, p0_values, marker='o', label='Percentile 0')
plt.plot(workgroupSizesX, p5_values, marker='o', label='Percentile 5')

# Logarithmic X axis
plt.xscale('log', base=2)
plt.xticks(workgroupSizesX, [str(x) for x in workgroupSizesX])

ax = plt.gca()
ax.yaxis.set_major_locator(MultipleLocator(200))

plt.xlabel('Workgroup Size X')
plt.ylabel('Time, ms')
plt.title('Benchmark')
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
python_kernel.destroy_device_buffer(inDevice)
python_kernel.destroy_device_buffer(rDevice)
python_kernel.destroy()